# LinkedIn Job Description Generation



## Section 1 — Setup



In [1]:
import re
import random
from pathlib import Path
import math
import json
import time
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from transformers import (
    AutoTokenizer,
    T5ForConditionalGeneration,
    get_linear_schedule_with_warmup,
)
from sklearn.model_selection import train_test_split
from langdetect import detect, DetectorFactory
DetectorFactory.seed = 0 
from tqdm.auto import tqdm
tqdm.pandas()

In [27]:
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DATA_DIR = Path("./data")                             
CHECKPOINT_DIR = Path("./checkpoint")                  
CHECKPOINT_SCRATCH_DIR = Path("./checkpoint_scratch") 

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
MODEL_NAME = "google/flan-t5-small"

GEN_KWARGS = dict(
    max_length=384,
    num_beams=4,
    no_repeat_ngram_size=3,
    early_stopping=True,
)

## Section 2 — Load and clean data

Loads the raw CSVs and builds a clean DataFrame.

In [28]:
CACHE_PATH = DATA_DIR / "df_cleaned.csv"

if CACHE_PATH.exists():
    df = pd.read_csv(CACHE_PATH)
    print(f"found a file in the cache with {len(df):,} rows")
else:
    df = None
    print("no existing file, let's run and cache the file")


found a file in the cache with 91,971 rows


In [29]:
if df is None:
    POSTINGS_COLS = [
        "job_id",
        "title",
        "description",
        "location",
        "formatted_work_type",
        "formatted_experience_level",
    ]
    
    postings = pd.read_csv(DATA_DIR / "postings.csv", usecols=POSTINGS_COLS)
    print(f"looking at postings.csv: {len(postings):,} rows")
    

In [30]:
if df is None:
    postings = postings.dropna(subset=["title", "description"]).reset_index(drop=True)
    print(f"after dropping missing title/description: {len(postings):,} rows")
    
    word_counts = postings["description"].str.split().str.len()
    before = len(postings)
    postings = postings[(word_counts >= 50) & (word_counts <= 700)].reset_index(drop=True)
    print(f"After 50-700 word filter: {len(postings):,}")

In [31]:
if df is None:
    def is_english(text: str) -> bool:
        try:
            return detect(str(text)[:500]) == "en"
        except Exception:
            return False
    
    print("looking only for english descriptions")
    mask_en = postings["description"].progress_apply(is_english)
    before = len(postings)
    postings = postings[mask_en].reset_index(drop=True)
    print(f"after english-only filter: {len(postings):,}")

In [32]:
if df is None:
    job_skills = pd.read_csv(DATA_DIR / "job_skills.csv")
    skills     = pd.read_csv(DATA_DIR / "skills.csv")
    
    skills_joined = (
        job_skills.merge(skills, on="skill_abr", how="left")
                  .dropna(subset=["skill_name"])
                  .groupby("job_id")["skill_name"]
                  .apply(lambda s: ", ".join(sorted(set(s))))
                  .rename("skills_joined")
                  .reset_index()
    )
    print(f"Skill rows: {len(job_skills):,}  ->  unique jobs with skills: {len(skills_joined):,}")
    
    job_industries = pd.read_csv(DATA_DIR / "job_industries.csv")
    industries     = pd.read_csv(DATA_DIR / "industries.csv")
    
    industries_joined = (
        job_industries.merge(industries, on="industry_id", how="left")
                      .dropna(subset=["industry_name"])
                      .groupby("job_id")["industry_name"]
                      .apply(lambda s: ", ".join(sorted(set(s))))
                      .rename("industries_joined")
                      .reset_index()
    )
    print(f"Industry rows: {len(job_industries):,}  ->  unique jobs with industries: {len(industries_joined):,}")
    
    df = (
        postings
        .merge(skills_joined,     on="job_id", how="left")
        .merge(industries_joined, on="job_id", how="left")
    )
    print(f"\nFinal cleaned dataframe: {len(df):,} rows, {df.shape[1]} columns")
    print(df.columns.tolist())
    df.tail()
    
    df.to_csv(CACHE_PATH, index=False)
    print(f"Saved cache -> {CACHE_PATH}  ({CACHE_PATH.stat().st_size / 1e6:.1f} MB)")
    

## Section 3 Build prompts and targets


In [33]:
def _clean_text(s) -> str:
    """Collapse all whitespace runs into a single space. Keeps a multi-line
    field (e.g. a title with a stray newline) on one line, and is also used
    for the target description."""
    return re.sub(r"\s+", " ", str(s)).strip()

def _or_unspecified(value) -> str:
    """NaN / None / empty string -> 'not specified'. Otherwise the cleaned value."""
    if value is None:
        return "not specified"
    if isinstance(value, float) and pd.isna(value):
        return "not specified"
    s = _clean_text(value)
    return s if s else "not specified"

def build_prompt(row) -> str:
    """Build the model input string from a row of df.

    The exact format below is what the model is trained on; app.py must call
    this identical function so inference-time prompts match training prompts.
    """
    return (
        "generate a job description for the following role.\n"
        f"title: {_or_unspecified(row.get('title'))}\n"
        f"work_type: {_or_unspecified(row.get('formatted_work_type'))}\n"
        f"experience: {_or_unspecified(row.get('formatted_experience_level'))}\n"
        f"location: {_or_unspecified(row.get('location'))}\n"
        f"industry: {_or_unspecified(row.get('industries_joined'))}\n"
        f"skills: {_or_unspecified(row.get('skills_joined'))}"
    )

# Sanity check on the first row.
print(build_prompt(df.iloc[0]))

generate a job description for the following role.
title: Marketing Coordinator
work_type: Full-time
experience: not specified
location: Princeton, NJ
industry: Real Estate
skills: Marketing, Sales


In [34]:
n = len(df)
sampled = df.sample(n=n, random_state=SEED).reset_index(drop=True)
print(f"randomized and indicizzate {len(sampled):,} rows")

print("Building (prompt, target) pairs...")
sampled["prompt"] = sampled.apply(build_prompt, axis=1)
sampled["target"] = sampled["description"].apply(_clean_text)

train_df, temp_df = train_test_split(sampled, test_size=0.20, random_state=SEED)
val_df, test_df   = train_test_split(temp_df,  test_size=0.50, random_state=SEED)
train_df = train_df.reset_index(drop=True)
val_df   = val_df.reset_index(drop=True)
test_df  = test_df.reset_index(drop=True)
print(f"train={len(train_df):,}  val={len(val_df):,}  test={len(test_df):,}")

# Show 3 examples so we can sanity-check the format visually.
for i in range(3):
    print("=" * 80)
    print("PROMPT:")
    print(train_df.iloc[i]["prompt"])
    tgt = train_df.iloc[i]["target"]
    print("\nTARGET (first 300 chars):")
    print(tgt[:300] + ("..." if len(tgt) > 300 else ""))

randomized and indicizzate 91,971 rows
Building (prompt, target) pairs...
train=73,576  val=9,197  test=9,198
PROMPT:
generate a job description for the following role.
title: CDL Water Truck Driver - SunZia North
work_type: Full-time
experience: Entry level
location: Encino, NM
industry: Construction
skills: Management, Manufacturing

TARGET (first 300 chars):
A DAY IN THE LIFE Drives water truck to spray water on roadways in order to control dust.Sees that the right amount of water is used for the particular condition so as not to over or under water.Fills truck with water from supply, obtains alternate supply if normal source is not available.Maintains ...
PROMPT:
generate a job description for the following role.
title: Inside Sales / Customer Service
work_type: Full-time
experience: not specified
location: Wood Dale, IL
industry: Appliances, Electrical, and Electronics Manufacturing
skills: Business Development, Sales

TARGET (first 300 chars):
The Agency is looking for a motivate

## Section 4 Tokenize 



In [35]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
print(f"Tokenizer: {type(tokenizer).__name__}, vocab size: {tokenizer.vocab_size}")

MAX_INPUT_LEN  = 256
MAX_TARGET_LEN = 384

class JobPostingDataset(Dataset):
    def __init__(self, df, tokenizer, max_input_len=MAX_INPUT_LEN, max_target_len=MAX_TARGET_LEN):
        prompts = df["prompt"].tolist()
        targets = df["target"].tolist()

        print(f"  Pre-tokenizing {len(prompts):,} examples...", end=" ", flush=True)
        enc_in = tokenizer(
            prompts,
            max_length=max_input_len,
            padding="max_length",
            truncation=True,
            return_tensors="pt",
        )
        enc_out = tokenizer(
            targets,
            max_length=max_target_len,
            padding="max_length",
            truncation=True,
            return_tensors="pt",
        )
        self.input_ids      = enc_in["input_ids"]
        self.attention_mask = enc_in["attention_mask"]
        labels = enc_out["input_ids"].clone()
        labels[labels == tokenizer.pad_token_id] = -100
        self.labels = labels
        print("done.")

    def __len__(self):
        return len(self.input_ids)

    def __getitem__(self, idx):
        return {
            "input_ids":      self.input_ids[idx],
            "attention_mask": self.attention_mask[idx],
            "labels":         self.labels[idx],
        }

Tokenizer: T5Tokenizer, vocab size: 32100


In [36]:
print("Building tokenized splits to have them all pre tokenized for faster training")
train_ds = JobPostingDataset(train_df, tokenizer)
val_ds   = JobPostingDataset(val_df,   tokenizer)
test_ds  = JobPostingDataset(test_df,  tokenizer)

Building tokenized splits to have them all pre tokenized for faster training
  Pre-tokenizing 73,576 examples... done.
  Pre-tokenizing 9,197 examples... done.
  Pre-tokenizing 9,198 examples... done.


In [37]:
BATCH_SIZE = 8

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, pin_memory=(DEVICE.type == "cuda"))
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False, pin_memory=(DEVICE.type == "cuda"))
test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE, shuffle=False, pin_memory=(DEVICE.type == "cuda"))

## Section 5 Fine-tune FLAN-T5-small


In [ ]:
N_EPOCHS         = 3
_continuing      = (CHECKPOINT_DIR / "config.json").exists()
LR               = 3e-5 if _continuing else 1e-4
WEIGHT_DECAY     = 0.01
WARMUP_STEPS     = 500
GRAD_CLIP        = 1.0

total_steps = len(train_loader) * N_EPOCHS

USE_AMP = False #optimization for my gtx 1660 super used for training, no way i am able to do this without this, taken from documentation!

def evaluate(model, loader):
    model.eval()
    total, count = 0.0, 0
    with torch.no_grad():
        for batch in tqdm(loader, desc="val", leave=False):
            batch = {k: v.to(DEVICE, non_blocking=True) for k, v in batch.items()}
            out = model(**batch)
            total += out.loss.item() * batch["input_ids"].size(0)
            count += batch["input_ids"].size(0)
    return total / max(count, 1)

def train_model(checkpoint_dir, model, tag):
    checkpoint_dir = Path(checkpoint_dir)
    best_path = checkpoint_dir / "best_val.json"
    best_val = float("inf")
    if best_path.exists():
        best_val = json.loads(best_path.read_text())["best_val"]

    optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    scheduler = get_linear_schedule_with_warmup(
        optimizer, num_warmup_steps=WARMUP_STEPS, num_training_steps=total_steps
    )

    for epoch in range(N_EPOCHS):
        model.train()
        running_loss = 0.0
        t0 = time.time()
        pbar = tqdm(train_loader, desc=f"[{tag}] epoch {epoch+1}/{N_EPOCHS}")
        optimizer.zero_grad(set_to_none=True)

        for step, batch in enumerate(pbar):
            batch = {k: v.to(DEVICE, non_blocking=True) for k, v in batch.items()}
            # **batch unpacks dict into keyword args, equivalent to
            # model(input_ids=batch["input_ids"], attention_mask=..., labels=...)
            out  = model(**batch)
            loss = out.loss
            loss.backward()
            running_loss += loss.item()

            torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
            optimizer.step()
            scheduler.step()
            optimizer.zero_grad(set_to_none=True)

            if step > 0 and step % 100 == 0:
                pbar.set_postfix(
                    loss=f"{running_loss/(step+1):.4f}",
                    lr=f"{scheduler.get_last_lr()[0]:.2e}",
                )

        avg_train = running_loss / len(train_loader)
        val_loss  = evaluate(model, val_loader)
        dt = time.time() - t0
        print(f"[{tag}] epoch {epoch+1}: train_loss={avg_train:.4f}  val_loss={val_loss:.4f}  ({dt/60:.1f} min)")

        if val_loss < best_val:
            best_val = val_loss
            model.save_pretrained(checkpoint_dir)
            tokenizer.save_pretrained(checkpoint_dir)
            best_path.write_text(json.dumps({"best_val": best_val, "epoch": epoch + 1}))
            print(f"[{tag}]   saved (new best val_loss={best_val:.4f}) -> {checkpoint_dir}")

    return best_val

if _continuing:
    print(f"Continuing from fine-tuned checkpoint in {CHECKPOINT_DIR} (lr={LR:.0e}) ...")
    model_pre = T5ForConditionalGeneration.from_pretrained(CHECKPOINT_DIR).to(DEVICE)
else:
    print(f"Fine-tuning from scratch: {MODEL_NAME} (lr={LR:.0e}) ...")
    model_pre = T5ForConditionalGeneration.from_pretrained(MODEL_NAME).to(DEVICE)
model_pre.config.use_cache = False

best_pre = train_model(CHECKPOINT_DIR, model_pre, tag="pretrained")
print(f"Pretrained fine-tune done. Best val loss: {best_pre:.4f}")

del model_pre
if DEVICE.type == "cuda":
    torch.cuda.empty_cache()

Continuing from fine-tuned checkpoint in checkpoint (lr=3e-05) ...


Loading weights:   0%|          | 0/190 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


[pretrained] epoch 1/3:   0%|          | 0/9197 [00:00<?, ?it/s]

KeyboardInterrupt: 

In [40]:
model_eval = T5ForConditionalGeneration.from_pretrained(CHECKPOINT_DIR).to(DEVICE)
model_eval.config.use_cache = False
test_loss = evaluate(model_eval, test_loader)
print(f"Test loss: {test_loss:.4f}")
del model_eval
if DEVICE.type == "cuda":
    torch.cuda.empty_cache()

Loading weights:   0%|          | 0/190 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


val:   0%|          | 0/1150 [00:00<?, ?it/s]

Test loss: 2.7886


## Section 6 Character Transformer from scratch

In [44]:
N_TRAIN_S6 = 2000
train_s6 = train_df.iloc[:N_TRAIN_S6].reset_index(drop=True)

charset = set()
for _, row in train_s6.iterrows():
    charset.update(row["prompt"] + chr(10) + row["description"])

vocabulary = sorted(charset)
stoi = {c: i for i, c in enumerate(vocabulary)}
itos = {i: c for i, c in enumerate(vocabulary)}
vocab_size = len(vocabulary)

print(f"{vocab_size} unique characters")
print(vocabulary)

194 unique characters
['\x02', '\n', ' ', '!', '"', '#', '$', '%', '&', "'", '(', ')', '*', '+', ',', '-', '.', '/', '0', '1', '2', '3', '4', '5', '6', '7', '8', '9', ':', ';', '<', '=', '>', '?', '@', 'A', 'B', 'C', 'D', 'E', 'F', 'G', 'H', 'I', 'J', 'K', 'L', 'M', 'N', 'O', 'P', 'Q', 'R', 'S', 'T', 'U', 'V', 'W', 'X', 'Y', 'Z', '[', '\\', ']', '^', '_', 'a', 'b', 'c', 'd', 'e', 'f', 'g', 'h', 'i', 'j', 'k', 'l', 'm', 'n', 'o', 'p', 'q', 'r', 's', 't', 'u', 'v', 'w', 'x', 'y', 'z', '{', '|', '}', '~', '\x80', '\x8f', '\x92', '\x99', '\x9c', '\x9d', '¡', '§', '©', '®', '°', '²', '·', '»', '¿', 'Ø', 'á', 'â', 'ç', 'é', 'í', 'ñ', 'ó', 'ö', 'ü', 'ē', 'ę', 'ʼ', '\u2002', '\u200d', '‐', '‑', '–', '—', '‘', '’', '“', '”', '•', '…', '\u202f', '\u2060', '€', '™', '−', '●', '⚡', '✍', '✨', '\uf0b7', '️', '\ufeff', '�', '𝐋', '𝐑', '𝐚', '𝐜', '𝐞', '𝐢', '𝐥', '𝐧', '𝐨', '𝐭', '🌊', '🌍', '🌖', '🌟', '🎓', '🏆', '🏔', '🏠', '🐢', '🐬', '🐾', '👁', '👤', '👦', '👧', '👨', '👩', '💪', '💰', '💲', '💸', '💻', '📈', '📊', '📍', '📝',

### 6.1 Sliding window over concatenated text

In [47]:
CONTEXT_LEN = 256
STRIDE= 64

full_text = chr(10).join(
    row["prompt"] + chr(10) + row["description"]
    for _, row in train_s6.iterrows()
)
print(len(full_text))
encoded = [stoi.get(c, 0) for c in full_text]

xs = []
ys = []
for i in range(0, len(encoded) - CONTEXT_LEN, STRIDE):
    xs.append(encoded[i : i + CONTEXT_LEN])
    ys.append(encoded[i + 1 : i + CONTEXT_LEN + 1])

xs_tensor = torch.tensor(xs, dtype=torch.long)
ys_tensor = torch.tensor(ys, dtype=torch.long)

x_train_s6, x_test_s6, y_train_s6, y_test_s6 = train_test_split(
    xs_tensor, ys_tensor, test_size=0.2, random_state=SEED)
print(f"Train: {x_train_s6.shape[0]:,}   Test: {x_test_s6.shape[0]:,}")

6178535
Train: 77,228   Test: 19,308


### 6.2 Model embedding and decoder

In [50]:
class TransformerEmbedding(nn.Module):
    def __init__(self, embedding_dim, vocab_size, sequence_len):
        super().__init__()
        self.token_embedding    = nn.Embedding(vocab_size, embedding_dim)
        self.position_embedding = nn.Embedding(sequence_len, embedding_dim)
        self.register_buffer("positions", torch.arange(sequence_len).reshape(1, -1))

    def forward(self, x):
        seq_len = x.shape[1]
        return (self.token_embedding(x)
                + self.position_embedding(self.positions[:, :seq_len]))

class TransformerDecoder(nn.Module):
    def __init__(self, vocab_size, sequence_len, embedding_dim,
                 n_layer, dropout, mlp_dim, nhead):
        super().__init__()
        self.embedding = TransformerEmbedding(embedding_dim, vocab_size, sequence_len)
        block = nn.TransformerEncoderLayer(
            d_model=embedding_dim, nhead=nhead,
            dim_feedforward=mlp_dim, dropout=dropout, batch_first=True,
        )
        self.transformer = nn.TransformerEncoder(block, num_layers=n_layer)
        self.classifier  = nn.Linear(embedding_dim, vocab_size)

    def forward(self, x):
        seq_len = x.shape[1]
        x = self.embedding(x)
        causal_mask = nn.Transformer.generate_square_subsequent_mask(seq_len).to(x.device)
        x = self.transformer(x, mask=causal_mask, is_causal=True)
        return self.classifier(x)

In [51]:
EMBEDDING_DIM = 256
N_HEADS       = 8
N_LAYER       = 4
DROPOUT       = 0.2
MLP_DIM       = 512

transformer_char = TransformerDecoder(
    vocab_size=vocab_size,
    sequence_len=CONTEXT_LEN,
    embedding_dim=EMBEDDING_DIM,
    n_layer=N_LAYER,
    dropout=DROPOUT,
    mlp_dim=MLP_DIM,
    nhead=N_HEADS,
)

def sample_top_p(logits, p=0.9, temperature=1.0):
    temperature = max(temperature, 1e-5)
    scaled_logits = logits / temperature
    sorted_logits, sorted_indices = torch.sort(scaled_logits, descending=True)
    cumulative_probs = torch.cumsum(F.softmax(sorted_logits, dim=-1), dim=-1)
    sorted_indices_to_remove = cumulative_probs > p
    sorted_indices_to_remove[..., 1:] = sorted_indices_to_remove[..., :-1].clone()
    sorted_indices_to_remove[..., 0] = False
    filtered_logits = scaled_logits.clone()
    indices_to_remove = sorted_indices_to_remove.scatter(
        0, sorted_indices, sorted_indices_to_remove
    )
    filtered_logits[indices_to_remove] = float("-inf")
    probs = F.softmax(filtered_logits, dim=-1)
    return torch.multinomial(probs, num_samples=1)

def generate_text_S6(model, initial_text="", temperature=1.0,
                     p=0.9, length=400, device=DEVICE):
    model.eval()
    encoded_text = [stoi[c] for c in initial_text if c in stoi]
    if not encoded_text:
        encoded_text = [stoi[vocabulary[0]]]
    with torch.no_grad():
        while len(encoded_text) < length:
            context = encoded_text[-CONTEXT_LEN:]
            x = torch.tensor(context, dtype=torch.long, device=device).reshape(1, -1)
            y = model(x)
            logits = y[0, -1, :].to("cpu")
            next_token = sample_top_p(logits, p=p, temperature=temperature)
            encoded_text.append(next_token.item())
    return "".join(itos[i] for i in encoded_text)


_sample_prompt = build_prompt(train_s6.iloc[0])

In [ ]:
EPOCHS_S6         = 10
BATCH_SIZE_S6     = 64
BATCHES_PER_EPOCH = 500
LEARNING_RATE_S6  = 3e-4

transformer_char.to(DEVICE)
criterion_S6 = nn.CrossEntropyLoss()
optimizer_S6 = torch.optim.Adam(transformer_char.parameters(), lr=LEARNING_RATE_S6)

pbar = tqdm(range(EPOCHS_S6), desc="epoch")
for epoch in pbar:
    rand_idxs = torch.randperm(x_train_s6.shape[0])
    x_perm    = x_train_s6[rand_idxs]
    y_perm    = y_train_s6[rand_idxs]

    print(generate_text_S6(transformer_char, _sample_prompt, length=200))

    transformer_char.train()
    losses = []
    for batch_idx, idx in enumerate(range(0, len(x_perm), BATCH_SIZE_S6)):
        if batch_idx >= BATCHES_PER_EPOCH:
            break
        x_batch = x_perm[idx : idx + BATCH_SIZE_S6].to(DEVICE)
        y_batch = y_perm[idx : idx + BATCH_SIZE_S6].to(DEVICE)

        optimizer_S6.zero_grad()
        preds = transformer_char(x_batch)
        loss  = criterion_S6(preds.permute(0, 2, 1), y_batch)
        loss.backward()
        optimizer_S6.step()

        losses.append(loss.item())
        pbar.set_description(f"epoch {epoch + 1} loss = {sum(losses)/len(losses):.4f}")

TRANSFORMER_PATH = CHECKPOINT_SCRATCH_DIR / "char_transformer.pt"
torch.save(transformer_char.state_dict(), TRANSFORMER_PATH)

_vocab_data = {
    "itos":          vocabulary,
    "context_len":   CONTEXT_LEN,
    "embedding_dim": EMBEDDING_DIM,
    "n_heads":       N_HEADS,
    "n_layer":       N_LAYER,
    "dropout":       DROPOUT,
    "mlp_dim":       MLP_DIM,
}
(CHECKPOINT_SCRATCH_DIR / "char_vocab.json").write_text(json.dumps(_vocab_data))
print(f"Saved model -> {TRANSFORMER_PATH}")
print(f"Saved vocab -> {CHECKPOINT_SCRATCH_DIR / chr(39)}char_vocab.json{chr(39)}")

Compare models

In [52]:
# --- [1/2] T5 models -------------------------------------------------------
@torch.no_grad()
def generate_t5(model, prompt: str) -> str:
    enc = tokenizer(prompt, max_length=MAX_INPUT_LEN, truncation=True,
                    return_tensors="pt").to(DEVICE)
    return tokenizer.decode(model.generate(**enc, **GEN_KWARGS)[0],
                            skip_special_tokens=True)

print("Loading fine-tuned FLAN-T5 checkpoint...")
model_pre_eval = T5ForConditionalGeneration.from_pretrained(CHECKPOINT_DIR).to(DEVICE).eval()
print("Done.\n")

# --- [3] Char-level Transformer (Section 6) --------------------------------
_na          = "(model not available – run Section 6 first)"
generate_char = None

_transf_path = CHECKPOINT_SCRATCH_DIR / "char_transformer.pt"
_vocab_path  = CHECKPOINT_SCRATCH_DIR / "char_vocab.json"

if _transf_path.exists() and _vocab_path.exists():
    _vd        = json.loads(_vocab_path.read_text())
    _itos_list = _vd["itos"]
    _stoi_s8   = {c: i for i, c in enumerate(_itos_list)}
    _itos_s8   = {i: c for i, c in enumerate(_itos_list)}
    _VOCAB_S8  = len(_itos_list)
    _CTX_S8    = _vd["context_len"]

    _transf_s8 = TransformerDecoder(
        vocab_size=_VOCAB_S8,  sequence_len=_CTX_S8,
        embedding_dim=_vd["embedding_dim"], n_layer=_vd["n_layer"],
        dropout=_vd.get("dropout", 0.2),   mlp_dim=_vd["mlp_dim"],
        nhead=_vd["n_heads"],
    )
    _transf_s8.load_state_dict(torch.load(str(_transf_path), map_location=DEVICE))
    _transf_s8.to(DEVICE).eval()
    print("Char Transformer loaded from disk.\n")

    @torch.no_grad()
    def generate_char(prompt: str, max_new: int = 400,
                      temperature: float = 0.8) -> str:
        seed = [_stoi_s8.get(c, 0) for c in prompt + "\n"]
        for _ in range(max_new):
            context = seed[-_CTX_S8:]
            x = torch.tensor(context, dtype=torch.long, device=DEVICE).unsqueeze(0)
            logits = _transf_s8(x)[0, -1, :]
            probs  = torch.softmax(logits / max(temperature, 1e-6), dim=-1).cpu().numpy()
            next_id = int(np.random.choice(_VOCAB_S8, p=probs))
            seed.append(next_id)
        return "".join(_itos_s8.get(i, "") for i in seed[len(prompt) + 1:])

# --- Comparison -------------------------------------------------------------
sample_row    = val_df.sample(1, random_state=SEED).iloc[0]
sample_prompt = build_prompt(sample_row)

print("Prompt:\n", sample_prompt, "\n")
print("Ground truth (first 400 chars):\n", sample_row["description"][:400], "\n")

base_model_s8 = T5ForConditionalGeneration.from_pretrained(MODEL_NAME).to(DEVICE).eval()

print("=" * 60)
print("[1] Zero-shot FLAN-T5-small:")
print(generate_t5(base_model_s8, sample_prompt))
print()
print("=" * 60)
print("[2] Fine-tuned FLAN-T5-small (Section 5):")
print(generate_t5(model_pre_eval, sample_prompt))
print()
print("=" * 60)
print("[3] Char-level Transformer (Section 6):")
print(generate_char(sample_prompt) if generate_char else _na)

Loading fine-tuned FLAN-T5 checkpoint...


Loading weights:   0%|          | 0/190 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


Done.

Char Transformer loaded from disk.

Prompt:
 generate a job description for the following role.
title: OPERATIONS ASSISTANT MANAGER
work_type: Full-time
experience: Mid-Senior level
location: Lexington, NC
industry: Retail
skills: Management, Manufacturing 

Ground truth (first 400 chars):
 Store Dollar Tree

Work where you love to shop! Dollar Tree is hiring in your neighborhood. Avoid long commutes and set your own course to success by applying today.

We offer generous benefits, flexible work schedules and the ability to work today and get paid tomorrow.

Responsible for assisting with all operational tasks within the store as delegated and assigned by the Store Manager with main  



Loading weights:   0%|          | 0/190 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


[1] Zero-shot FLAN-T5-small:
ASSISTANT MANAGER is a mid-senior level position in the retail industry in Lexington, NC.

[2] Fine-tuned FLAN-T5-small (Section 5):
Store Dollar Tree Work where you love to shop! Dollar Tree is hiring in your neighborhood. Avoid long commutes and set your own course to success by applying today. We offer generous benefits, flexible work schedules and the ability to work today and get paid tomorrow. As a Dollar Tree Assistant Store Manager you will be responsible for providing exceptional service to our customers. A key priority includes assisting the Store Manager in the daily operation of the store. Under the direction of the Store Assistant Manager, you will also assist with the hiring, training and development of store associates. Principal Duties & Responsibilities: Greets and assists customers in a positive, approachable manner. Answers questions and resolves customer inquiries and concerns. Assists in ensuring a clean, well-stocked store for customer